# UK Road Accidents — Narrative Analysis

Companion notebook to `road-accidents` — walks through the EDA and modeling story from the class project, now calling into the `src/road_accidents/` package rather than reproducing the code inline.

**Prereqs:** run `make prepare` (produces `data/processed/*.parquet` + `models/encoders.joblib`) and `make train` (produces `models/{lr,rf,xgb}.joblib` + `reports/results.json`) before executing this notebook.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from road_accidents.config import CLASS_NAMES, FIGURES_DIR, MODELS_DIR, PROCESSED_DIR, REPORTS_DIR

## 1. Data

1.5M UK Department for Transport records (2005–2018). After dropping identifier/geo/admin columns and rows with any missing values, we're left with ~1.5M rows and 14 pre-crash features (before feature engineering). Post-encoding we have 64 numeric features.

Class distribution is heavily skewed toward Slight, with Fatal being ~60x rarer — which drives most of the modeling headaches later.

In [ ]:
X_train = pd.read_parquet(PROCESSED_DIR / 'X_train.parquet')
X_test = pd.read_parquet(PROCESSED_DIR / 'X_test.parquet')
y_train = pd.read_parquet(PROCESSED_DIR / 'y_train.parquet')['y'].to_numpy()
y_test = pd.read_parquet(PROCESSED_DIR / 'y_test.parquet')['y'].to_numpy()

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train class balance (post-downsample): {pd.Series(y_train).value_counts().to_dict()}')
print(f'Test class balance: {pd.Series(y_test).value_counts().to_dict()}')

## 2. Correlation heatmap

The strongest correlation between numeric predictors is `Speed_limit` vs `Urban_or_Rural_Area` at ~0.68 — no multicollinearity concerns beyond that.

In [ ]:
from IPython.display import Image
Image(str(FIGURES_DIR / 'correlation_heatmap.png'))

## 3. Model results

`scripts/train_models.py` produces the JSON below. Macro-F1 is the primary metric — accuracy is misleading given the imbalance.

In [ ]:
results = json.loads((REPORTS_DIR / 'results.json').read_text())
rows = []
for r in results:
    row = {'model': r['name'], 'macro_f1': r['macro_f1'], 'accuracy': r['accuracy']}
    for i, cls in enumerate(CLASS_NAMES):
        row[f'F1_{cls}'] = r['per_class_f1'][i]
    rows.append(row)
pd.DataFrame(rows).set_index('model').round(3)

### Logistic regression coefficients

Top features by mean absolute coefficient across classes. Speed limit dominates. Time-of-day (cyclical hour, rush hour) and number of vehicles carry meaningful weight too.

In [ ]:
Image(str(FIGURES_DIR / 'lr_coefficients.png'))

### Random Forest permutation importance

RF leans heavily on `Number_of_Vehicles` — shuffling it destroys performance more than any other feature.

In [ ]:
Image(str(FIGURES_DIR / 'rf_permutation.png'))

### XGBoost SHAP summary (Fatal class)

Speed limit and number of vehicles are the dominant drivers of Fatal-class predictions in XGBoost. Cyclical hour and urban/rural are secondary.

In [ ]:
Image(str(FIGURES_DIR / 'xgb_shap_fatal.png'))

## 4. Conclusion

Across all three models, **speed limit** and **number of vehicles** are the strongest signals for accident severity. XGBoost narrowly wins on macro-F1 (~0.36) but all three models struggle to identify the Fatal class — reflecting both the extreme class imbalance and the limited discriminative power of the pre-crash features in this dataset.

See the README's *What I'd do next* section for the follow-ups I'd try to push fatal-class recall past its current ceiling.